In [1]:
import os
import torch 
import pandas as pd
from torchvision import models 
from torchinfo import summary
from pathlib import Path

from model import init_model

In [2]:
ROOT_DIR = Path(os.path.dirname(os.path.abspath('')))
TOTAL_TRAIN_SAMPLES = 50
TOTAL_TEST_SAMPLES = 10
CHANNELS = 3
BATCH_SIZE = 16
DEVICE = "cuda:1"
OCT_PRESENCE = "Usando OCT"
DUAL_IMAGE = "Dual Image"
DATA_PATH = "../data.csv"
SUMMARY_PATH = "../model_summary.csv"
HISTORY_PATH = "../history_csv"
FT_SIZE = 24
OUTPUT_TAB = 5

In [3]:
usage_mem = pd.read_csv("../usage_mem_gpu.csv")

In [4]:
models_list =  [
            "regnetx",
            "regnet16x",
            "regnet32x",
            "mobile",
            "shuffle",
            "efficient",
            "vit",
            "inception",
            "resnet",
            "regnet",
            "regnet16",
            "regnet32",
        ]


In [11]:
def get_memory(backbone: str, double_img: bool, output_tab: int):
    model, input_size = init_model(backbone, True, double_img, output_tab, FT_SIZE)
    if not double_img and not output_tab:
        res = summary(model,  device = 'cuda:1', input_size=(BATCH_SIZE,CHANNELS,input_size,input_size))
    else:
        res = summary(model, device = 'cuda:1', input_size=[(BATCH_SIZE,CHANNELS,input_size,input_size),(BATCH_SIZE,CHANNELS,input_size,input_size),(BATCH_SIZE, FT_SIZE)])
    total_bytes = res.to_megabytes(res.total_input + res.total_output_bytes + res.total_param_bytes)
    return int(total_bytes)

In [12]:
list_of_tuples = []
for model in models_list:
    res_single = get_memory(model, False, 0)
    res_single_tab = get_memory(model, False, OUTPUT_TAB)
    res_double = get_memory(model, True, 0)
    res_double_tab = get_memory(model, True, OUTPUT_TAB)
    list_of_tuples.append((model, BATCH_SIZE, 0, 0, res_single))
    list_of_tuples.append((model, BATCH_SIZE, 0, OUTPUT_TAB, res_single_tab))
    list_of_tuples.append((model, BATCH_SIZE, 1, 0, res_double))
    list_of_tuples.append((model, BATCH_SIZE, 1, OUTPUT_TAB, res_double_tab))

/scratch/felipemarcelino/glaucoma/envs/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=RegNet_X_1_6GF_Weights.IMAGENET1K_V1`. You can also use `weights=RegNet_X_1_6GF_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/scratch/felipemarcelino/glaucoma/envs/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=RegNet_X_3_2GF_Weights.IMAGENET1K_V1`. You can also use `weights=RegNet_X_3_2GF_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/scratch/felipemarcelino/glaucoma/envs/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments ot

In [13]:
df = pd.DataFrame(list_of_tuples, columns=['backbone', 'batch_size', 'double_img','output_tab','mem_usage_torchinfo'])
df

,backbone,batch_size,double_img,output_tab,mem_usage_torchinfo
0,regnetx,16,0,0,1353
1,regnetx,16,0,5,1364
2,regnetx,16,1,0,2712
3,regnetx,16,1,5,2712
4,regnet16x,16,0,0,2073
5,regnet16x,16,0,5,2085
6,regnet16x,16,1,0,4155
7,regnet16x,16,1,5,4155
8,regnet32x,16,0,0,2976
9,regnet32x,16,0,5,2988


In [14]:
usage_mem

,backbone,batch_size,mem_usage,output_tab,double_img
0,regnet,16,1800,0,0
1,regnetx,16,1706,0,0
2,regnet16,16,2358,0,0
3,regnet16x,16,2130,0,0
4,regnet32,16,3182,0,0
5,regnet32x,16,2848,0,0
6,inception,16,3384,0,0
7,resnet,16,2700,0,0
8,vit,16,4864,0,0
9,efficient,16,2378,0,0


In [15]:
mem_usage_general = df.merge(usage_mem)
mem_usage_general

,backbone,batch_size,double_img,output_tab,mem_usage_torchinfo,mem_usage
0,regnetx,16,0,0,1353,1706
1,regnetx,16,0,5,1364,1712
2,regnetx,16,1,0,2712,2446
3,regnetx,16,1,5,2712,2446
4,regnet16x,16,0,0,2073,2130
5,regnet16x,16,0,5,2085,2148
6,regnet16x,16,1,0,4155,3284
7,regnet16x,16,1,5,4155,3284
8,regnet32x,16,0,0,2976,2848
9,regnet32x,16,0,5,2988,2890


In [16]:
mem_usage_general["diff_mem"] = mem_usage_general["mem_usage"] - mem_usage_general["mem_usage_torchinfo"]

In [17]:
mem_usage_general

,backbone,batch_size,double_img,output_tab,mem_usage_torchinfo,mem_usage,diff_mem
0,regnetx,16,0,0,1353,1706,353
1,regnetx,16,0,5,1364,1712,348
2,regnetx,16,1,0,2712,2446,-266
3,regnetx,16,1,5,2712,2446,-266
4,regnet16x,16,0,0,2073,2130,57
5,regnet16x,16,0,5,2085,2148,63
6,regnet16x,16,1,0,4155,3284,-871
7,regnet16x,16,1,5,4155,3284,-871
8,regnet32x,16,0,0,2976,2848,-128
9,regnet32x,16,0,5,2988,2890,-98


In [62]:
mem_usage_general.to_csv("../mem_diff_cu117",index=False)